# Polynomial regression

In [1]:
import warnings
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

## Data that follows a curve: y = 0.8x² + 0.9x + 2 + noise

In [2]:
rng = np.random.default_rng(42)
X = 6 * rng.random((200, 1)) - 3
y = (0.8 * X ** 2 + 0.9 * X + 2 + rng.standard_normal((200, 1))).ravel()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
lr = LinearRegression().fit(X_train, y_train)
print("straight line, test R2", round(r2_score(y_test, lr.predict(X_test)), 3))

straight line, test R2 0.378


## Add an x² column

In [3]:
poly = PolynomialFeatures(degree=2, include_bias=True)
X_train_p = poly.fit_transform(X_train)
X_test_p = poly.transform(X_test)
print(X_train[0], "->", X_train_p[0].round(3))     # [x] -> [1, x, x^2]
lr2 = LinearRegression().fit(X_train_p, y_train)
print("degree 2, test R2", round(r2_score(y_test, lr2.predict(X_test_p)), 3))
print("coefficients", lr2.coef_.round(3), "intercept", round(lr2.intercept_, 3))   # close to 0.9, 0.8 and 2

[-0.10617938] -> [ 1.    -0.106  0.011]
degree 2, test R2 0.826
coefficients [0.    1.044 0.825] intercept 1.921


## Interactive: drag the slider to change the degree (25 training points)

In [4]:
Xs = 6 * rng.random((25, 1)) - 3
ys = (0.8 * Xs ** 2 + 0.9 * Xs + 2 + rng.standard_normal((25, 1))).ravel()
grid = np.linspace(-3, 3, 300).reshape(-1, 1)
degrees = list(range(1, 16))
fig = go.Figure([go.Scatter(x=Xs.ravel(), y=ys, mode="markers", name="training points")])
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d, include_bias=False), StandardScaler(), LinearRegression()).fit(Xs, ys)
    fig.add_scatter(x=grid.ravel(), y=np.clip(m.predict(grid), -5, 15), mode="lines", name=f"degree {d}", visible=(d == 2))
steps = [{"label": str(d), "method": "update", "args": [{"visible": [True] + [dd == d for dd in degrees]}]} for d in degrees]
fig.update_layout(sliders=[{"steps": steps, "active": 1, "currentvalue": {"prefix": "degree: "}}], yaxis_range=[-5, 15])
fig

## Two inputs: the new columns include x², y² and xy

In [5]:
p = PolynomialFeatures(degree=2).fit(np.zeros((1, 2)))
print(p.get_feature_names_out(["x", "y"]))
print("degree 30 with 2 inputs gives", PolynomialFeatures(30).fit(np.zeros((1, 2))).n_output_features_, "columns")

['1' 'x' 'y' 'x^2' 'x y' 'y^2']
degree 30 with 2 inputs gives 496 columns


In [6]:
x = 7 * rng.random(100) - 2.8
yy = 7 * rng.random(100) - 2.8
z = x ** 2 + yy ** 2 + 0.2 * x + 0.2 * yy + 0.1 * x * yy + 2 + rng.standard_normal(100)
XY = np.c_[x, yy]                    # two columns side by side (not np.array([x, y]).reshape(100, 2))
print("plane R2   ", round(LinearRegression().fit(XY, z).score(XY, z), 3))
print("degree 2 R2", round(make_pipeline(PolynomialFeatures(2), LinearRegression()).fit(XY, z).score(XY, z), 3))

plane R2    0.381
degree 2 R2 0.981


In [7]:
# Test: is the gap between the learned and true coefficients due to the noise?
y_clean = (0.8 * X_train ** 2 + 0.9 * X_train + 2).ravel()      # same rows, noise removed
m0 = LinearRegression().fit(X_train_p, y_clean)
print("no noise: coefficients", m0.coef_.round(3), "intercept", round(m0.intercept_, 3))

no noise: coefficients [0.  0.9 0.8] intercept 2.0


In [8]:
# Test: degree 15 on 25 points. Is the wild curve a numerical error or overfitting?
from numpy.polynomial import Legendre
rng2 = np.random.default_rng(42)
f = lambda x: 0.8 * x ** 2 + 0.9 * x + 2
X_tr = 6 * rng2.random((25, 1)) - 3; y_tr = f(X_tr).ravel() + rng2.standard_normal(25)
X_te = 6 * rng2.random((200, 1)) - 3; y_te = f(X_te).ravel() + rng2.standard_normal(200)
P = PolynomialFeatures(15, include_bias=False).fit_transform(X_tr)
print("condition number raw", f"{np.linalg.cond(np.c_[np.ones(25), P]):.1e}",
      "scaled", f"{np.linalg.cond(np.c_[np.ones(25), StandardScaler().fit_transform(P)]):.1e}")
pipe = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler(), LinearRegression()).fit(X_tr, y_tr)
leg = Legendre.fit(X_tr.ravel(), y_tr, 15, domain=[-3, 3])     # same model, well-conditioned basis
print("pipeline  train R2", round(pipe.score(X_tr, y_tr), 4), "test R2", round(pipe.score(X_te, y_te), 2))
print("Legendre  train R2", round(r2_score(y_tr, leg(X_tr.ravel())), 4), "test R2", round(r2_score(y_te, leg(X_te.ravel())), 1))

condition number raw 4.5e+08 scaled 1.1e+06


pipeline  train R2 0.9917 test R2 -9.35
Legendre  train R2 0.9932 test R2 -3726.7
